# Notebook 02 — Why Is My Customer Lookup So Slow?
## Clustering: Organizing Data for Non-Date Queries

**The scenario**: Your monthly report by date is fast (Notebook 01 showed why). But when you look up a specific customer's transactions, it scans the entire table. Why?

**The answer**: The table is ordered by date — but you're filtering by `account_id`. Snowflake must check every partition because accounts are scattered everywhere.

**The fix**: A clustered table organizes data by `account_id`, so lookups only scan a handful of partitions.

| Table | Organized By | Account Lookup Behavior |
|-------|-------------|------------------------|
| `TRANSACTIONS` | date (natural order) | Full scan — accounts scattered across all partitions |
| `TRANSACTIONS_CLUSTERED` | account_id | Pruned — each account's data in a few partitions |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL_V2.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Worst Case: Account Lookup on Date-Ordered Table

"Show me all transactions for account ACC-0000012345."

In [ ]:
-- WORST: Date-ordered table — account_id scattered across ALL partitions
SELECT
    COUNT(*) AS txn_count,
    MIN(transaction_date) AS first_txn,
    MAX(transaction_date) AS last_txn
FROM TRANSACTIONS
WHERE account_id = 'ACC-0000012345';

---
## Best Case: Account Lookup on Clustered Table

In [ ]:
-- BEST: Clustered by account_id — only scans partitions containing this account
SELECT
    COUNT(*) AS txn_count,
    MIN(transaction_date) AS first_txn,
    MAX(transaction_date) AS last_txn
FROM TRANSACTIONS_CLUSTERED
WHERE account_id = 'ACC-0000012345';

---
## See the Difference: Clustering Metadata

In [ ]:
-- How well is account_id clustered in each table?
SELECT 'TRANSACTIONS (date-ordered)' AS table_name, 
       PARSE_JSON(SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS', '(account_id)')) AS clustering_info
UNION ALL
SELECT 'TRANSACTIONS_CLUSTERED', 
       PARSE_JSON(SYSTEM$CLUSTERING_INFORMATION('TRANSACTIONS_CLUSTERED', '(account_id)'));

---
## Compare the Results

In [ ]:
-- Compare bytes scanned and time
SELECT
    CASE
        WHEN query_text ILIKE '%TRANSACTIONS_CLUSTERED%' THEN 'CLUSTERED (best)'
        ELSE 'UNCLUSTERED (worst)'
    END AS approach,
    bytes_scanned / (1024*1024*1024) AS gb_scanned,
    total_elapsed_time / 1000 AS elapsed_sec
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_WAREHOUSE(
    WAREHOUSE_NAME => 'HOL_XS',
    END_TIME_RANGE_START => DATEADD(hour, -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 50
))
WHERE query_text ILIKE '%ACC-0000012345%'
  AND query_text NOT ILIKE '%QUERY_HISTORY%'
  AND query_text NOT ILIKE '%CLUSTERING_INFORMATION%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 2;

---
## Key Takeaway

| Metric | Unclustered | Clustered | Improvement |
|--------|------------|-----------|-------------|
| GB Scanned | ~4-5 GB | ~0.01 GB | **500x less** |
| Time | ~5-10 sec | <1 sec | **10x+ faster** |

**Why it matters for you as an analyst:**
- If you frequently filter by a column that ISN'T the date (customer ID, account ID, region), ask your data engineer about clustering
- Clustering is automatic once set up — Snowflake maintains it as new data arrives
- You can check clustering health yourself: `SELECT SYSTEM$CLUSTERING_INFORMATION('table', '(column)')`

**When to request clustering:**
- Your queries consistently filter on the same non-date column
- The table has billions of rows
- Your queries scan way more data than expected

**Next** → [Notebook 03 — Search Optimization](./Notebook_03_Search_Optimization.ipynb) (for needle-in-a-haystack lookups)